In [2]:
import math
import sys
from pathlib import Path

_here = Path().resolve()
for _p in [_here, *_here.parents]:
    _src = _p / "src"
    if (_src / "qudits_on_qubits" / "__init__.py").is_file():
        repo_root = _p
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
else:
    raise ImportError(
        "qudits_on_qubits repo root not found; run the notebook from notebooks/ or the repo root"
    )

from qiskit.quantum_info import Statevector, Operator, partial_trace, SparsePauliOp
from qiskit import qpy, QuantumCircuit
import numpy as np
from qiskit.synthesis import TwoQubitWeylDecomposition
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, Batch

from qudits_on_qubits import create_ame_circuit, generate_b_ame
from sympy.functions.combinatorial.numbers import legendre_symbol
from IPython.display import display, Math
from itertools import product
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from igraph import Graph, plot
import matplotlib.pyplot as plt

from qudits_on_qubits.bell_measurements.sampler_circuits import build_sampler_circuits_from_graph, build_sampler_circuits_for_candidate
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
from iqm.iqm_client import CircuitCompilationOptions, DDMode, STANDARD_DD_STRATEGY, DDStrategy
from iqm.qiskit_iqm import IQMProvider
from iqm.iqm_client.transpile import ExistingMoveHandlingOptions
from iqm.qubit_selector.qubit_selector import CostEvaluator
from iqm.qubit_selector.qiskit_utils import perform_backend_transpilation

from qudits_on_qubits.bell_measurements.sampler_circuits import run_sampler_circuits_to_counts_by_setting
from qudits_on_qubits.bell_measurements.sampler_circuits import decoding_kwargs_from_metadata
from qudits_on_qubits.bell_measurements.postprocessing import compute_bell_value_from_counts

In [3]:
dd_options = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=STANDARD_DD_STRATEGY,
  )

dd_default = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED)

dd_xy4 = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=DDStrategy(
          gate_sequences=[(5, "YXYX", "asap")]
      ),
  )

dd_xy8 = CircuitCompilationOptions(dd_mode=DDMode.ENABLED, 
                                   dd_strategy=DDStrategy(
                                       gate_sequences=[(9, 'XYXYYXYX', 'center')]))

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
provider_garnet = IQMProvider("https://resonance.iqm.tech/", quantum_computer="garnet")
backend_garnet = provider_garnet.get_backend(use_metrics=True)


In [8]:
from provider import get_backend, get_backend_error_profile, generate_random_error_profile, to_static_architecture
from iqm.qiskit_iqm.fake_backends.iqm_fake_backend import IQMFakeBackend
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet

In [13]:
backend = get_backend(quantum_computer="garnet")
error_profile = generate_random_error_profile(backend=backend_garnet)

garnet = IQMFakeGarnet()
garnet_architecture = garnet.architecture

static_garnet_architecture = to_static_architecture(garnet_architecture)

static_garnet_architecture = to_static_architecture(garnet_architecture)

garnet_noisy_backend = IQMFakeBackend(architecture=static_garnet_architecture, error_profile=error_profile)

Backend connection failed, using fake backend, using IQMFakeAphroditeBackend

Noise profile: random-noise-profile

T1 / T2 / readout
qubit   T1 [us]   T2 [us]  readout 0->1  readout 1->0  readout avg error  readout asymmetry
  QB1 25.386361 22.847725      0.046494      0.043153           0.044824          -0.003340
  QB2 60.299919 23.257612      0.038258      0.034737           0.036497          -0.003521
  QB3 63.703094 45.654582      0.030012      0.039112           0.034562           0.009100
  QB4 72.876619 40.614473      0.018911      0.026765           0.022838           0.007854
  QB5 55.841533 25.788296      0.022057      0.022778           0.022417           0.000721
  QB6 26.717139 24.045425      0.027921      0.024566           0.026244          -0.003355
  QB7 33.942548 30.548293      0.040870      0.050141           0.045506           0.009271
  QB8 57.224449 22.318318      0.018162      0.021842           0.020002           0.003679
  QB9 29.278028 26.350226      0.061788

In [14]:
def fold_cz_preserve_layout(qc: QuantumCircuit, n: int = 3) -> QuantumCircuit:
      if n < 1:
          raise ValueError("n musi byc >= 1")
      if n % 2 == 0:
          raise ValueError("Do ZNE uzyj nieparzystego n: 1, 3, 5, ...")

      out = qc.copy()
      old_data = list(out.data)
      out.data.clear()

      for inst in old_data:
          op = inst.operation
          qargs = inst.qubits
          cargs = inst.clbits

          if op.name.lower() == "cz":
              for _ in range(n):
                  out.append(op.copy(), qargs, cargs)
          else:
              out.append(op.copy(), qargs, cargs)

      out.name = f"{qc.name}_czfold{n}"
      return out

In [15]:
import mthree

In [16]:
ghz_best_list = ["sup012_P012_ph022", "sup012_P012_ph121", "sup012_P012_ph221", "sup012_P012_ph222", "sup012_P021_ph012", "sup012_P021_ph022", "sup012_P021_ph212", "sup012_P021_ph222", "sup123_P120_ph222", "sup123_P210_ph121", "sup123_P210_ph222"]

def load_candidate(candidate):
    selected_candidate = f"monomial_full__{candidate}"
    #QuditsOnQubits\artifacts\iqm_runs\selected_best\two_qutrit\stage2_top10_rerun20_20260706\exact\rank01_monomial_full__sup023_P012_ph022
    artifact_circuit_dir = repo_root / "artifacts" / "iqm_runs" / "raw" / "quantum_circuits" / "garnet" / "ghz3_qpy13" / selected_candidate
    legacy_circuit_dir = repo_root.parent / "QuditsOnQubits" / "basis_direct_encoding_benchmarks" / "quantum_circuits" / "ghz3_qpy13" / selected_candidate

    for circuit_dir in (artifact_circuit_dir, legacy_circuit_dir):
        if (circuit_dir / "graph_state_direct_basis.qpy").is_file():
            break
    else:
        raise FileNotFoundError(
            "Circuit artifacts not found. Checked:\n"
            f"- {artifact_circuit_dir}\n"
            f"- {legacy_circuit_dir}"
        )

    print(f"Loading circuits from: {circuit_dir}")

    with (circuit_dir / "graph_state_direct_basis.qpy").open("rb") as f:
        testqc = qpy.load(f)[0]

    with (circuit_dir / "graph_state_direct_basis_transpiled.qpy").open("rb") as f:
        qcsuptrans = qpy.load(f)[0]

    with (circuit_dir / "F3_W.qpy").open("rb") as f:
        F3sup = qpy.load(f)[0]

    Esup = np.load(circuit_dir / "E.npy")

    return testqc, qcsuptrans, F3sup, Esup


def compact_pretranspiled_state(circuit):
    """Relabel an already synthesized physical circuit as a 6-qubit logical circuit."""
    if circuit.layout is None:
        raise ValueError("Pretranspiled circuit has no final layout metadata")

    final_layout = circuit.layout.final_index_layout(filter_ancillas=True)
    if not final_layout or len(set(final_layout)) != len(final_layout):
        raise ValueError("Pretranspiled circuit has an invalid final layout")

    physical_to_logical = {
        physical: logical for logical, physical in enumerate(final_layout)
    }
    compact = QuantumCircuit(len(final_layout), name=f"{circuit.name}_logical")
    for instruction in circuit.data:
        if instruction.clbits:
            raise ValueError("State-preparation circuit must not contain classical bits")
        physical_qubits = [
            circuit.find_bit(qubit).index for qubit in instruction.qubits
        ]
        if not all(index in physical_to_logical for index in physical_qubits):
            raise ValueError(
                f"Operation {instruction.operation.name} uses a qubit outside final layout"
            )
        compact.append(
            instruction.operation.copy(),
            [compact.qubits[physical_to_logical[index]] for index in physical_qubits],
        )
    return compact

In [17]:
def select_and_transpile_candidate(
    transpile_backend,
    logical_state_circuit,
    sampler_circuits,
    candidate,
):
    layouts, costs = CostEvaluator(
        backend=transpile_backend,
        quantum_circuit=logical_state_circuit,
    ).get_top_layouts(num_layouts=1)
    if not layouts:
        raise RuntimeError(
            f"IQM Qubit Selector returned no valid layout for {candidate}"
        )

    best_layout = list(layouts[0])
    reduced_coupling_map = transpile_backend.coupling_map.reduce(mapping=best_layout)
    transpiled_circuits = perform_backend_transpilation(
        sampler_circuits,
        transpile_backend,
        best_layout,
        reduced_coupling_map,
        qiskit_optim_level=3,
    )
    return transpiled_circuits, best_layout, float(costs[0])


In [18]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

READOUT_CACHE_VERSION = 1
DEFAULT_READOUT_CACHE_PATH = (
    repo_root / "artifacts" / "iqm_runs" / "calibration" / "readout_matrices.json"
)


def _backend_cache_key(backend):
    client = getattr(backend, "client", None)
    server_client = getattr(client, "_iqm_server_client", None)
    root_url = getattr(server_client, "root_url", None)
    quantum_computer = getattr(server_client, "_quantum_computer", None)
    if isinstance(root_url, str) and isinstance(quantum_computer, str):
        return f"{root_url.rstrip('/')}::{quantum_computer}"

    name = backend.name
    return str(name() if callable(name) else name)


def _load_readout_cache(cache_path):
    if not cache_path.exists():
        return {"version": READOUT_CACHE_VERSION, "backends": {}}

    try:
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(
            f"Cannot read readout calibration cache: {cache_path}"
        ) from exc

    if (
        payload.get("version") != READOUT_CACHE_VERSION
        or not isinstance(payload.get("backends"), dict)
    ):
        raise ValueError(f"Invalid readout calibration cache: {cache_path}")
    return payload


def _matrix_from_cache(entry, backend_key, qubit):
    try:
        matrix = np.asarray(entry["matrix"], dtype=np.float32)
    except (KeyError, TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid cached matrix for {backend_key} qubit {qubit}"
        ) from exc

    if matrix.shape != (2, 2) or not np.isfinite(matrix).all():
        raise ValueError(f"Invalid cached matrix for {backend_key} qubit {qubit}")
    return matrix


def build_readout_calibration_matrices(
    backend,
    physical_qubits,
    shots=10_000,
    verbose=True,
    cache_path=None,
    force_recalibration=False,
):
    """
    Return an M3-compatible list of calibration matrices.

    Saved matrices are reused per backend and physical qubit. Only missing
    entries, or entries explicitly forced to refresh, run on the backend.
    """
    cache_path = Path(cache_path or DEFAULT_READOUT_CACHE_PATH)
    cache = _load_readout_cache(cache_path)
    backend_key = _backend_cache_key(backend)
    backend_cache = cache["backends"].setdefault(backend_key, {})
    cache_updated = False
    matrices = [None] * backend.num_qubits

    for q in sorted(set(physical_qubits)):
        if not isinstance(q, int) or not 0 <= q < backend.num_qubits:
            raise ValueError(f"Invalid physical qubit index: {q}")

        cached_entry = backend_cache.get(str(q))
        if cached_entry is not None and not force_recalibration:
            matrices[q] = _matrix_from_cache(cached_entry, backend_key, q)
            if verbose:
                print(f"Qubit {q} (cache):")
                print(matrices[q])
            continue

        cal_0 = QuantumCircuit(backend.num_qubits, 1)
        cal_0.measure(q, 0)

        cal_1 = QuantumCircuit(backend.num_qubits, 1)
        cal_1.x(q)
        cal_1.measure(q, 0)

        counts_0, counts_1 = backend.run(
            [cal_0, cal_1],
            shots=shots,
        ).result().get_counts()

        p10 = counts_0.get("1", 0) / shots
        p01 = counts_1.get("0", 0) / shots
        matrices[q] = np.array(
            [
                [1 - p10, p01],
                [p10, 1 - p01],
            ],
            dtype=np.float32,
        )
        backend_cache[str(q)] = {
            "matrix": matrices[q].tolist(),
            "shots": shots,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
        cache_updated = True
        if verbose:
            print(f"Qubit {q}:")
            print(matrices[q])

    if cache_updated:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = cache_path.with_name(f"{cache_path.name}.tmp")
        temporary_path.write_text(
            json.dumps(cache, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        os.replace(temporary_path, cache_path)

    return matrices


In [19]:
def apply_mitigation(counts_by_setting, mit, mapping):
    quasi = []

    for res in list(counts_by_setting.values()):
        quasi.append(mit.apply_correction(res, mapping, return_mitigation_overhead=True))

    quasi_miti = {}

    for i, setting in zip(quasi, list(counts_by_setting.keys())):
        quasi_temp = {}
        for key in i.keys():
            quasi_temp[key] = int(i[key]*1024*20)
        quasi_miti[setting] = quasi_temp

    return quasi_miti

In [20]:
import pandas as pd

In [25]:
def full_pipeline(
    transpile_backend,
    execution_backend,
    logical_state_circuit,
    Esup,
    candidate,
    rows=None,
):
    if rows is None:
        rows = []

    qutrit_qubits = ((0, 1), (2, 3), (4, 5))
    sampler_circuits, metadata = build_sampler_circuits_for_candidate(
        candidate="ghz3",
        state_circuit=logical_state_circuit,
        E=Esup,
        qutrit_qubits=qutrit_qubits,
    )
    isa_sampler_qc, selected_layout, selector_cost = select_and_transpile_candidate(
        transpile_backend, logical_state_circuit, sampler_circuits, candidate
    )

    isa_sampler_qc_1 = isa_sampler_qc
    transpiled_depth = isa_sampler_qc_1[0].depth()
    transpiled_cz_count = isa_sampler_qc_1[0].count_ops().get("cz", 0)
    mapping = mthree.utils.final_measurement_mapping(isa_sampler_qc_1[0])
    expected_classical_bits = set(range(6))
    if set(mapping) != expected_classical_bits:
        raise RuntimeError(
            f"Expected measurements for classical bits 0..5, got {mapping}"
        )
    if not set(mapping.values()).issubset(set(selected_layout)):
        raise RuntimeError(
            f"Measured qubits {mapping} are inconsistent with layout {selected_layout}"
        )
    print(
        f"{candidate}: layout={selected_layout}, measurements={mapping}, "
        f"depth={transpiled_depth}, cz={transpiled_cz_count}"
    )
    isa_sampler_qc_3 = [fold_cz_preserve_layout(qc, n=3) for qc in isa_sampler_qc]
    isa_sampler_qc_5 = [fold_cz_preserve_layout(qc, n=5) for qc in isa_sampler_qc]

    counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(
        isa_sampler_qc_1,
        metadata,
        shots=1024 * 20,
        transpile_circuits=False,
        backend=execution_backend,
    )
    counts_by_setting3, run_info3 = run_sampler_circuits_to_counts_by_setting(
        isa_sampler_qc_3,
        metadata,
        shots=1024 * 20,
        transpile_circuits=False,
        backend=execution_backend,
    )
    counts_by_setting5, run_info5 = run_sampler_circuits_to_counts_by_setting(
        isa_sampler_qc_5,
        metadata,
        shots=1024 * 20,
        transpile_circuits=False,
        backend=execution_backend,
    )

    decoding_kwargs = decoding_kwargs_from_metadata(metadata)
    bell_value = compute_bell_value_from_counts(
        counts_by_setting,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )
    bell_value3 = compute_bell_value_from_counts(
        counts_by_setting3,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )
    bell_value5 = compute_bell_value_from_counts(
        counts_by_setting5,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )

    scale_factors = np.array([1.0, 3.0, 5.0])
    bell_values = np.array(
        [bell_value.real, bell_value3.real, bell_value5.real], dtype=float
    )
    coeff = np.polyfit(scale_factors, bell_values, deg=1)
    zne_linear = np.polyval(coeff, 0.0)

    measured_physical_qubits = sorted(set(mapping.values()))
    calibration_matrix = build_readout_calibration_matrices(
        execution_backend,
        measured_physical_qubits,
        shots=10_000,
        verbose=False,
        cache_path=DEFAULT_READOUT_CACHE_PATH,
    )

    mit = mthree.M3Mitigation()
    mit.cals_from_matrices(calibration_matrix)

    counts_by_setting_garnet_miti = apply_mitigation(
        counts_by_setting, mit, mapping
    )
    counts_by_setting_garnet_miti3 = apply_mitigation(
        counts_by_setting3, mit, mapping
    )
    counts_by_setting_garnet_miti5 = apply_mitigation(
        counts_by_setting5, mit, mapping
    )

    mitigated_counts = [
        counts_by_setting_garnet_miti,
        counts_by_setting_garnet_miti3,
        counts_by_setting_garnet_miti5,
    ]
    for counts in mitigated_counts:
        for setting_counts in counts.values():
            for key, value in setting_counts.items():
                if value < 0:
                    setting_counts[key] = 0

    bell_value_miti = compute_bell_value_from_counts(
        counts_by_setting_garnet_miti,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )
    bell_value_miti3 = compute_bell_value_from_counts(
        counts_by_setting_garnet_miti3,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )
    bell_value_miti5 = compute_bell_value_from_counts(
        counts_by_setting_garnet_miti5,
        metadata["terms"],
        metadata["qutrit_bit_indices_by_setting"],
        **decoding_kwargs,
    )

    mitigated_bell_values = np.array(
        [bell_value_miti.real, bell_value_miti3.real, bell_value_miti5.real],
        dtype=float,
    )
    coeff = np.polyfit(scale_factors, mitigated_bell_values, deg=1)
    zne_linear_meas_miti = np.polyval(coeff, 0.0)

    rows.append(
        {
            "candidate": candidate,
            "selected_layout": tuple(selected_layout),
            "selector_cost": selector_cost,
            "measured_physical_qubits": tuple(measured_physical_qubits),
            "transpiled_depth": isa_sampler_qc_1[0].depth(),
            "transpiled_cz_count": isa_sampler_qc_1[0].count_ops().get("cz", 0),
            "zne_linear": zne_linear,
            "bell_value": bell_value.real,
            "bell_value3": bell_value3.real,
            "bell_value5": bell_value5.real,
            "zne_linear_meas_miti": zne_linear_meas_miti,
            "bell_value_miti": bell_value_miti.real,
            "bell_value_miti3": bell_value_miti3.real,
            "bell_value_miti5": bell_value_miti5.real,
        }
    )
    return rows


In [29]:
backend

In [26]:
garnet

In [27]:
garnet_noisy_backend

In [30]:
rows = []
for candidate in ghz_best_list:
    testqc, qcsuptrans, F3sup, Esup = load_candidate(candidate)
    compact_qc = compact_pretranspiled_state(qcsuptrans)
    rows = full_pipeline(
        backend_garnet, garnet_noisy_backend, compact_qc, Esup,
        candidate=candidate, rows=rows,
    )

df = pd.DataFrame(rows)
df


Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\ghz3_qpy13\monomial_full__sup012_P012_ph022


c:\Users\szymo\anaconda3\envs\qudityD3_laptop\Lib\site-packages\qiskit\qpy\interface.py:316: UserWarning: The qiskit version used to generate the provided QPY file, 2.5.0, is newer than the current qiskit version 2.1.2. This may result in an error if the QPY file uses instructions not present in this current qiskit version
  warnings.warn(


[07-13 16:54:56;I] Number of layouts to evaluate: 14
[07-13 16:55:01;I] Cost evaluation has begun using cost function "gate_cost_cz".
Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\ghz3_qpy13\monomial_full__sup012_P012_ph121
[07-13 16:55:20;I] Number of layouts to evaluate: 14
[07-13 16:55:25;I] Cost evaluation has begun using cost function "gate_cost_cz".
Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\ghz3_qpy13\monomial_full__sup012_P012_ph221
[07-13 16:55:41;I] Number of layouts to evaluate: 14
[07-13 16:55:46;I] Cost evaluation has begun using cost function "gate_cost_cz".
Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\ghz3_qpy13\monomial_full__sup012_P012_ph222
[07-13 16:56:03;I] Number of layouts to evaluate: 14
[07-13 16:56:09;I] Cost evaluation has begun using cost function "ga

,candidate,selected_layout,selector_cost,measured_physical_qubits,zne_linear,bell_value,bell_value3,bell_value5,zne_linear_meas_miti,bell_value_miti,bell_value_miti3,bell_value_miti5
0,sup012_P012_ph022,"(2, 3, 7, 8, 12, 13)",0.943036,"(2, 3, 7, 8, 12, 13)",0.030548,0.016634,-0.014472,-0.041645,0.051146,0.028751,-0.015799,-0.060637
1,sup012_P012_ph121,"(2, 3, 7, 8, 12, 13)",0.920882,"(2, 3, 7, 8, 12, 13)",-0.016223,-0.013261,-0.032717,-0.021715,-0.022460,-0.018774,-0.043032,-0.029332
2,sup012_P012_ph221,"(2, 3, 7, 8, 12, 13)",0.943462,"(2, 3, 7, 8, 12, 13)",0.091394,0.065870,0.033315,-0.021432,0.123429,0.089617,0.047327,-0.025363
3,sup012_P012_ph222,"(2, 3, 7, 8, 12, 13)",0.941552,"(2, 3, 7, 8, 12, 13)",-0.016304,-0.008914,0.027846,0.038230,-0.022276,-0.013207,0.037552,0.049165
4,sup012_P021_ph012,"(3, 4, 7, 8, 9, 12, 13)",0.921487,"(3, 7, 8, 9, 12, 13)",-0.012565,-0.017093,0.012876,-0.003986,-0.014822,-0.022547,0.021465,-0.005877
5,sup012_P021_ph022,"(2, 3, 7, 8, 12, 13)",0.937135,"(2, 3, 7, 8, 12, 13)",0.025081,0.011144,0.043556,0.003624,0.038486,0.017421,0.062691,0.003079
6,sup012_P021_ph212,"(2, 3, 7, 8, 12, 13)",0.950020,"(2, 3, 7, 8, 12, 13)",0.057378,0.038645,-0.022423,-0.055168,0.078135,0.052792,-0.032281,-0.076091
7,sup012_P021_ph222,"(2, 3, 7, 8, 12, 13)",0.946883,"(2, 3, 7, 8, 12, 13)",-0.022503,-0.015954,0.000251,0.012728,-0.028801,-0.020394,0.002035,0.017727
8,sup123_P120_ph222,"(2, 3, 7, 8, 9, 12, 13)",0.936751,"(3, 7, 8, 9, 12, 13)",0.007766,0.000115,0.048657,0.020585,0.007552,-0.004705,0.068658,0.024569
9,sup123_P210_ph121,"(3, 4, 5, 6, 8, 9, 10, 11)",0.957853,"(3, 4, 5, 6, 8, 9)",0.020056,0.003073,0.001443,-0.038990,0.031912,0.007512,0.003876,-0.053958
